In [1]:
#!/usr/bin/env python3
# =============================================================================
#  LoViF 2026 — AEIC-ME board-aligned FINETUNE (training track)  •  Team HackFleet
#  RUN TARGET: Kaggle RTX PRO 6000 (96 GB), INTERNET OFF.  Resumable (12h cap).
#  NO SCORE SACRIFICE: runs the FULL perceptual+adversarial recipe offline by
#  using staged assets (see lovif-stage-train-assets.py, run once on a T4).
# =============================================================================
#  WHY THIS FILE EXISTS
#  --------------------
#  stablecodec.ipynb (board=68.41) runs the two PUBLIC AEIC checkpoints
#  (SE_ft16 + ME_ft16) and picks SE-or-ME per image under the 0.008 bpp budget.
#  Its ceiling is fixed by those pretrained checkpoints, and the ENTIRE 3.8-pt
#  gap to #1 is perceptual: LPIPS 0.217 vs 0.069, DISTS 0.095 vs 0.033 (sunsean
#  even has LOWER PSNR/MS-SSIM than us — the lead is bought purely with LPIPS/DISTS).
#
#  The repo ships the official recipe to move exactly those numbers
#  (README "Finetune towards target bitrates with GAN", config/finetune_AEIC_ME.yaml,
#  src/finetune.py). Two facts in that recipe map straight onto the board:
#    1. `lambda_rate` sets the bitrate (ft16 = lambda_rate 16; HIGHER => LOWER bpp).
#       ft16's native rate is ABOVE 0.008 (that's why stablecodec must drop to SE
#       for most images). A checkpoint finetuned at a higher lambda_rate sits
#       natively at ~0.0075 bpp, so every image keeps the stronger ME-class decoder.
#    2. finetune.py weights DISTS only 1.0 and computes LPIPS at EVAL ONLY. The
#       board pays 25*DISTS + 20*LPIPS. So: RAISE lambda_dists and ADD an LPIPS
#       term -> training optimizes what is actually scored.
#
#  This script finetunes AEIC-ME WARM-STARTED from AEIC_ME_ft16.pkl and saves a
#  .pkl in the EXACT save_model() format, so it drops straight into the stablecodec
#  multi-rate pipeline as a new variant (genuine constriction bitstreams, leak-free,
#  no test-time tuning, decoder unchanged & <4 GB).
#
#  FULL OFFLINE LOSS (no score sacrifice) — equals finetune.py's recipe + LPIPS:
#      loss = lambda_rate * rate_bpp                         (TargetRateModule, diff'able)
#           + lambda_l2   * MSE                              (keeps PSNR/structure)
#           + lambda_dists* OC-EA-DISTS(DISTS_pytorch)       (board 25 -> push hard)
#           + lambda_lpips* LPIPS(lpips, alex)               (board 20 -> NEW term)
#           + lambda_clip * CLIP-cos     (optional, when CLIP ViT-B/32 is staged)
#           + lambda_adv  * hinge GAN(frozen DINOv2 + trainable head)   (vision-aided)
#  Faithfulness notes:
#    * OC-EA-DISTS replicates src/oc_losses/oc_ea_dists_loss.py (overlap-chunk to 224
#      + Sobel-edge DISTS, 1.5*d + 0.5*edge) but on DISTS_pytorch + the official
#      weights.pt — i.e. the SAME DISTS the board scores (NOT a pyiqa downgrade).
#    * The vision-aided GAN uses a FROZEN DINOv2-ViT-B/14 backbone + a from-scratch
#      conv head + hinge loss — the same mechanism as vision_aided_loss, but with a
#      non-gated, easily-staged backbone instead of license-gated DINOv3. This is the
#      adversarial-realism term; it is what we would otherwise have "sacrificed" offline.
#  The GAN auto-enables ONLY if the staged DINOv2 backbone is found; otherwise the
#  run still does the full perceptual loss (rate+L2+DISTS+LPIPS[+CLIP]) and just logs
#  that the adversarial term is off. Nothing crashes for a missing optional asset.
#
#  WHAT IS TRAINED / SAVED (4 GB test-phase cap satisfied)
#  ------------------------------------------------------
#  Trainable (AEIC_ME_Trainer.set_training_mode): full codec + UNet LoRA(r=32) +
#  unet.conv_in/conv_out. Frozen: base SD-Turbo UNet, VAE decoder, ALL loss backbones.
#  save_model() writes ONLY {unet lora/conv_in/conv_out, codec} -> small .pkl. The
#  deployed decoder is the SAME SD-Turbo stack stablecodec already ships.
# =============================================================================

import os
import gc
import re
import sys
import math
import time
import types
import random
import warnings
import importlib
import importlib.util
import importlib.abc
import importlib.machinery
import subprocess
import contextlib
from pathlib import Path

warnings.filterwarnings("ignore")

# =============================================================================
#  CONFIG  (edit these; everything else auto-detects)
# =============================================================================
ASSUME_OFFLINE = True          # RTX PRO 6000 = internet OFF. Staged datasets only.

#  SETTLED 2026-06-30 after 4 sweep rounds (see SWEEP_MODE below for full history): the initial
#  native-check favored ft32, but that compared ft32's untrained native numbers to an undertrained
#  ft16 -- unfair. A matched-depth 3x3 grid (round 2) showed ft16 beats BOTH ft8 and ft32 at every
#  matched lambda. ft16 confirmed as the warm-start.
WARMSTART_NAMES = ["AEIC_ME_ft16", "AEIC_ME_ft32", "AEIC_ME_ft8"]   # finetune FROM the first found

# ---- native-checkpoint check (2026-06-30) ----------------------------------
#  We've only ever warm-started from AEIC_ME_ft16.pkl. The public AEIC release ships FIVE
#  rate points (ft2/4/8/16/32 -- LOWER ft = LOWER lambda_rate = LESS downgrade = HIGHER
#  bpp/quality; HIGHER ft = MORE downgrade = LOWER bpp/quality). ft16 sits well above our
#  ~0.008 target natively (own eval at step 0 = 0.0131), so every run so far has had to push
#  a LARGE external lambda_rate (28->100) to drag it down -- a big jump from where ft16's
#  weights were originally optimized, plausibly behind the metric volatility seen mid-training.
#  ft32 was already finetuned at the AUTHORS' highest rate-penalty point; its native bpp is
#  presumably much closer to budget already, so warm-starting from it might need only a small
#  additional lambda_rate push and converge to better quality AT OUR TARGET BUDGET specifically
#  (not better quality in general -- ft2/ft4 have better unconstrained quality, but they're
#  further from where we need to end up). CKPT_CHECK_MODE runs ZERO training steps, just loads
#  each candidate and evaluates it as-is, to see which is the best starting point.
#  RESULT (logged above WARMSTART_NAMES): ft32 wins, sits under budget natively with better
#  DISTS than ft16-trained-down. Check done -- mode off; WARMSTART_NAMES now defaults to ft32.
CKPT_CHECK_MODE  = False        # True: evaluate native checkpoints (no training), then exit
CKPT_CHECK_NAMES = ["AEIC_ME_ft8", "AEIC_ME_ft16", "AEIC_ME_ft32"]   # which staged .pkl to probe
CKPT_CHECK_EVAL_N = 24          # > production EVAL_N=8 -> less noisy reading, matches SWEEP_EVAL_N

# ---- bitrate target -------------------------------------------------------
#  rate_loss == differentiable total bpp (TargetRateModule). Operating point is set
#  PURELY by LAMBDA_RATE: higher => lower bpp. ft16 used 16 and lands ABOVE 0.008.
#  Watch the eval `bpp` line; nudge LAMBDA_RATE: too-high bpp -> raise; over-smoothed
#  & far below target -> lower.
#
#  IMPORTANT (organizers confirmed): the 0.008 cap is on the DATASET AVERAGE only —
#  there is NO per-image bitrate limit. So this checkpoint does NOT have to sit under
#  0.008 for every image. Two valid strategies:
#    (A) aim the ME finetune at ~0.0075 average -> every image keeps the good ME, no
#        knapsack downgrades needed (simplest; TARGET_BPP=0.0075).
#    (B) aim slightly ABOVE budget (~0.0085-0.009, higher quality) and let the
#        stablecodec knapsack trim the few most-expensive images to a cheaper option
#        (SE_ft16 or a 2nd lower-rate finetune) so the AVERAGE hits 0.008 while the
#        MAJORITY keep the better reconstruction. Often scores higher. Needs a cheap
#        fallback in the knapsack menu.
#  DECIDED 2026-06-30: Strategy B, lambda_rate=70. We already lead #1 (sunsean) on PSNR/
#  MS-SSIM -- the entire score gap is LPIPS/DISTS, and every sweep round showed raising
#  lambda_rate DEGRADES exactly those two metrics. Round 4 (matched depth, NEW loss weights,
#  ft16): bpp/DISTS/LPIPS = 0.00989/.1704/.2804 (l=70), 0.00929/.1703/.2916 (l=85),
#  0.00844/.1746/.2961 (l=100). l=70 has the BEST LPIPS and (tied) best DISTS -- the exact
#  metrics we're chasing -- plus best PSNR. ALL THREE are OVER 0.008 (none is a standalone
#  fit), so all require the knapsack to mix regardless. Under Strategy B, "maximize the
#  budget" means handing the knapsack the HIGHEST-QUALITY option to allocate (it always uses
#  exactly 0.008 on avg no matter what); the knapsack is purpose-built to exploit an
#  over-budget high-quality checkpoint by downgrading only the cheapest images. So the best
#  budget utilization = best per-image quality = l=70, NOT the lower-bpp/lower-quality l=100.
#  (An earlier l=100 pick was a mistake: it optimized budget-FIT over the LPIPS/DISTS the user
#  explicitly prioritized, on an unverified "less SE downgrade" assumption that also contradicts
#  the Strategy-B mechanism. Reverted to l=70.)
LAMBDA_RATE  = 70.0            # SET from round-4 sweep: best LPIPS/DISTS (the priority metrics)
#  TWO-CHECKPOINT MENU (2026-06-30 user decision): train BOTH a high-quality l=70 variant and a
#  lower-rate l=100 variant, BOTH warm-started from ft16, in one launch. This gives stablecodec's
#  knapsack an intermediate rung so it downgrades expensive images in a SMALL step (l70->l100)
#  instead of cliff-diving to the much-worse SE codec -- the genuine budget-maximizer. Each lambda
#  produces its own deploy .pkl (AEIC_ME_lovif_ft70.pkl / ft100.pkl) and its own per-lambda resume
#  state (train_state_ft70.pt / ft100.pt), so a single run does both and survives the 12h cap.
#  Set to [LAMBDA_RATE] (single value) to revert to a one-checkpoint run.
FULL_RUN_LAMBDAS = [70.0, 100.0]
#  2026-06-30: raised from 0.0075 -> 0.0079. The goal is to USE the full 0.008 budget, not
#  conservatively undershoot it (a checkpoint sitting at e.g. 0.0064 wastes ~20% of the bits
#  we're allowed, i.e. real quality left on the table). 0.0079 leaves only a small margin for
#  (a) real constriction arithmetic-coding overhead vs the ideal entropy estimate measured here,
#  (b) sampling noise between our eval-crop subset and the full dataset average. For Strategy B
#  (recommended -- see below) this is a SOFT target per-checkpoint, not a hard ceiling: a
#  checkpoint that lands somewhat ABOVE 0.008 is fine and often BETTER, because the knapsack's
#  per-image adaptive allocation (not a uniform rate) is what actually maximizes budget usage --
#  cheap images get downgraded to a fallback, expensive ones keep the higher-quality variant,
#  and the GLOBAL average lands at exactly 0.008 instead of every image conservatively undershooting.
TARGET_BPP   = 0.0079          # logged reference only (knapsack enforces the AVERAGE budget)

# ---- rate sweep (2026-06-30 finding) ---------------------------------------
#  A full run at LAMBDA_RATE=28 plateaued at eval bpp ~0.0175 (2.3x over the 0.0075/0.008
#  target) within ~250 steps and never moved further in 5000 steps. The richer board-aligned
#  loss (added MSSSIM+LPIPS+CLIP+GAN on top of finetune.py's original L2+DISTS@1.0) pulls
#  noticeably harder toward quality than the original recipe did, so 28 isn't enough rate
#  pressure to land near budget anymore. Rather than guess-and-rerun full 5000-step jobs
#  (~30 min each), SWEEP_MODE runs several SHORT trials (fresh warm-start from the ORIGINAL
#  ft16 each time, not cumulative) across candidate lambdas and reports where each one's
#  bpp equilibrates, since the rate visibly converges within ~250 steps regardless of value.
#  DECISION MADE (2026-06-30): 4 sweep rounds done, see LAMBDA_RATE comment above for the final
#  call (ft16, lambda_rate=70, Strategy B). Sweep mode off -- next run is the real full job.
SWEEP_MODE         = False     # True: run the cheap multi-lambda scan instead of the full job
#  Round 1 (50/70/100, warm-started from ft16, 300 steps) gave a clean monotonic R-D curve:
#  50->0.01136bpp, 70->0.00925bpp, 100->0.00673bpp. A native-checkpoint check (CKPT_CHECK_MODE,
#  see above) then showed ft32 with ZERO training already beating "ft16 forced down via 300
#  steps" on DISTS at a similar bpp (native ft32: bpp .00639/DISTS .176 vs ft16+lambda100 after
#  300 steps: bpp .00673/DISTS .196). BUT that's not a fair test -- it compares ft32's native
#  (fully, separately trained by AEIC's authors) numbers to a possibly-undertrained ft16. So
#  SWEEP_WARMSTARTS tests checkpoints at the SAME (bumped) training depth, with the SAME lambda
#  candidates, for a genuinely fair side-by-side comparison instead of guessing.
#  Added ft8 (2026-06-30, user-flagged): its native numbers (CKPT_CHECK_MODE result) were
#  bpp=.02043, LPIPS=.208, DISTS=.122 -- the BEST native quality of ft8/16/32, by a clear margin.
#  Excluding it from the trained comparison would repeat the exact same unfairness already
#  caught with ft32 vs ft16 -- we have trustworthy native data for it, so it gets a fair shot too.
#  ft4/ft2 deliberately EXCLUDED (user decision, 2026-06-30): even less downgrade than ft8's
#  lambda_rate=8, so native bpp is presumably further still from the ~0.008 target -- a bigger
#  external lambda push would be needed than even ft8 required, with more uncertainty about
#  whether quality survives. Not tested at all (no native-check data either) -- treated as too
#  far from the target to be worth the compute, not as a confirmed loser.
#
#  ROUND 2 RESULT (2026-06-30, matched 600-step depth, ft8 vs ft16 vs ft32 x 50/70/100):
#  ft16 beats BOTH ft8 and ft32 at matched lambda -- e.g. lambda=70: ft16 bpp=.00909/DISTS=.1724
#  vs ft32 bpp=.00763/DISTS=.1921 (worse DISTS despite FEWER bits) vs ft8 bpp=.00993/DISTS=.1788
#  (worse DISTS despite MORE bits). The earlier ft32-wins conclusion was an artifact of comparing
#  ft32's untrained native numbers to ft16's undertrained (300-step) ones -- confirms the user's
#  original "shouldn't ft16 be better" instinct was right once given a fair, matched-depth test.
#  ft32 lambda=70 (bpp .00763) is still the best STANDALONE fit (only one safely under 0.008 with
#  reasonable quality), but ft16 lambda=70 (bpp .00909, ~14% over) has clearly better DISTS/LPIPS
#  -- worth it under Strategy B (knapsack absorbs the overshoot by downgrading the cheapest few
#  images). ROUND 3 refined lambda for ft16 ONLY (75/80/85): none got under 0.008 (closest was
#  85 at bpp .00818, +2.25% over) and quality degraded monotonically with lambda (DISTS .1744->
#  .1822 from 75->85) -- lambda=70 remained the best quality-per-bit point throughout, with a
#  rough knapsack-mixing estimate (assuming a ~0.0045bpp cheap fallback) suggesting ~76% of
#  images could still use it even at its +14% standalone overshoot.
#
#  ROUND 4 (2026-06-30, user directive): the loss weights just changed substantially -- fixed
#  LAMBDA_L2 (1.0->0.1, was a 10x over-weighting bug) and boosted LAMBDA_DISTS/LPIPS (2.5/2.0 ->
#  3.5/3.0) to chase the LPIPS/DISTS gap to #1 aggressively, per explicit instruction to accept
#  worse PSNR/MS-SSIM for it. ALL prior sweep data (rounds 1-3) was measured under the OLD
#  weights and is now stale -- re-testing the SAME lambdas (70/85/100) already have old data
#  points for, under the NEW weights, to see the reweighting's effect directly and isolate it
#  from the lambda effect.
SWEEP_WARMSTARTS   = ["AEIC_ME_ft16"]      # ft16 confirmed best warm-start (round 2) -- keep
SWEEP_LAMBDAS      = [70.0, 85.0, 100.0]   # reuse exact prior values -- isolates the reweighting's effect
#  bpp converges within ~250 steps and never moves again (confirmed in the ft28 run) -- but
#  QUALITY kept drifting well after that (LPIPS 0.270->0.225 over the FULL 5000 steps, DISTS
#  quietly got worse over the same span). 300 steps finds the right bpp but under-samples that
#  quality drift, which is exactly what we're ranking candidates on. Bumped to 600 (2x; still
#  ~12x cheaper than a full 5000-step run) to capture more of that trend without losing the
#  point of a cheap sweep. Subset bumped only enough to give headroom at 600 steps (600*BATCH=
#  1200 crops consumed, still < 2000 -> wasn't actually a constraint yet, but matters if steps
#  go higher later).
SWEEP_STEPS        = 600
SWEEP_TRAIN_SUBSET = 4000
SWEEP_EVAL_N       = 24        # > production EVAL_N=8 -> less noisy bpp/quality reading
SWEEP_LOG_EVERY    = 50

# ---- board-aligned loss weights -------------------------------------------
#  The board scores FOUR terms: PSNR(1) + 10*MS_SSIM + 20*(1-LPIPS) + 25*(1-DISTS).
#  Originally mirrored by scaling the board's own weights down by a constant /10 for
#  numerical convenience: MSSSIM=10/10=1.0, DISTS=25/10=2.5, LPIPS=20/10=2.0 -- but
#  LAMBDA_L2 was left at 1.0 instead of the proportionally-correct 1/10=0.1. That's a
#  10x OVER-weighting of the PSNR proxy relative to the other three terms' established
#  scaling -- anchoring the optimizer toward pixel fidelity more than the board rewards,
#  which works directly AGAINST closing the LPIPS/DISTS gap.
#  2026-06-30 (user directive): the entire 3.8-pt gap to #1 sunsean is perceptual --
#  sunsean has LOWER PSNR/MS-SSIM than us, so we have real headroom to trade some of
#  that advantage away for LPIPS/DISTS gains. Fixed L2 to the proportionally-correct
#  0.1, AND boosted DISTS/LPIPS further BEYOND proportional (1.4x/1.5x) to chase them
#  aggressively per that directive, while keeping their ratio (3.0/3.5=0.857) close to
#  the board's own 20/25=0.8 so neither dominates the other unnaturally. MSSSIM left at
#  the proportionally-correct 1.0 (board weight 10 isn't negligible, no reason given yet
#  to deprioritize it specifically beyond the natural board ratio).
#  RISK NOTE: pushing perceptual losses too far without a fidelity anchor can let the
#  network drift into texture/artifact hallucination that "games" LPIPS/DISTS (both are
#  themselves trained networks) without genuinely improving subjective quality -- L2=0.1
#  (not 0) and MSSSIM=1.0 are deliberately kept as a stabilizing floor, not removed.
LAMBDA_L2     = 0.1           # FIXED: was 1.0 (10x over-weighted vs the others' /10 scaling)
LAMBDA_MSSSIM = 1.0           # board weight 10/10 -- unchanged, already proportional
LAMBDA_DISTS  = 3.5           # board weight 25/10=2.5, boosted 1.4x -- chase aggressively
LAMBDA_LPIPS  = 3.0           # board weight 20/10=2.0, boosted 1.5x -- chase aggressively
#  LPIPS TRAINING-NET MIX (research-consensus fix, 2026-06-30). All three independent deep-research
#  reports flag the SAME thing: training DIRECTLY on LPIPS-Alex (the net the board scores) invites
#  metric-gaming and overfits the tiny 100-img dev set, hurting the hidden-2K final phase. The LPIPS
#  authors themselves state VGG is "closer to a traditional perceptual loss" for OPTIMIZATION while
#  Alex is the better forward METRIC. SPRDiff's loss (VGG perceptual) agrees. So: TRAIN on a VGG-
#  dominant blend, keep a SMALL Alex term so we still nudge the exact scored metric, and EVAL/select
#  on PURE Alex (= the board). These shares split the LPIPS term internally; LAMBDA_LPIPS scales the
#  whole. If the VGG backbone can't load offline, build_losses auto-falls back to Alex-only (the old
#  behavior) and logs it -- so this can never crash a run for a missing/odd VGG asset.
LPIPS_VGG_SHARE  = 0.8        # primary optimization driver (well-behaved gradients, harder to game)
LPIPS_ALEX_SHARE = 0.2        # small term toward the exact board metric (kept, not removed)
LAMBDA_CLIP   = 0.1           # recipe default; only applied if CLIP ViT-B/32 is staged
LAMBDA_ADV    = 0.1           # recipe default; only applied if DINOv2 backbone is staged
GAN_WARMUP    = 500           # linearly ramp adv weight 0->LAMBDA_ADV over N steps

# ---- optimization ---------------------------------------------------------
PATCH        = 512             # train crop (square). 96 GB handles 512 @ batch 2-4.
BATCH        = 2
LR           = 5e-5
LR_FLOOR     = 5e-6            # cosine-decayed to this
LR_DISC      = 2e-5
#  2026-06-30: 5000 -> 8000. At BATCH=2 that's 16k crops, ~1.1 passes over the 14,136-image set
#  (5000 was <1 epoch -- genuine under-training). The ft28 run showed quality kept improving over
#  the FULL span (LPIPS 0.270->0.225 across 5000 steps, still trending), so more steps help. ~45min
#  per lambda; resume survives the 12h cap, so the longer schedule costs nothing but wall-clock.
MAX_STEPS    = 8000            # optimizer steps (512-crop main phase). Resumable across 12h sessions.
GRAD_CLIP    = 0.1
SEED         = 123

# ---- high-res fine-tuning tail (research-consensus fix) --------------------
#  DISTS (board weight 25, our worst gap) computes texture covariance over WIDE receptive fields;
#  training only at 512 under-samples the broad structure of the 2K test images and leaves DISTS on
#  the table. After the 512 main phase, run a SHORT tail at 1024 (the 98 GB GPU handles it at small
#  batch) to broaden the statistics DISTS/LPIPS see -- exactly AEIC's own Stage-3 high-res-finetune
#  idea and what all three reports recommend. Runs at LR_FLOOR (lr_at floors past MAX_STEPS). The
#  tail is part of the SAME resumable step counter, so a mid-tail crash resumes correctly. 0=disable.
HIRES_TAIL_STEPS = 600         # extra 1024-crop steps appended after MAX_STEPS (0 to skip)
HIRES_PATCH      = 1024
HIRES_BATCH      = 1           # 1024 crops are 4x the pixels; keep batch small for headroom

# ---- checkpointing / eval (resume survives the 12h Kaggle cap) ------------
CKPT_EVERY   = 250
EVAL_EVERY   = 250
EVAL_N       = 8
LOG_EVERY    = 20
#  Select models on the SAME 8-bit PNG the real submission writes, not on float tensors: the
#  continuous->uint8 round-trip adds quantization noise that the live eval otherwise hides, so a
#  checkpoint can look better in-memory than it scores on the board. ON => honest model selection.
EVAL_PNG_ROUNDTRIP = True
OUT_DIR      = "/kaggle/working/aeic_ft_out"
TRAIN_SUBSET = None            # cap #train images for a fast smoke run (None = all)
# =============================================================================


_T0 = time.time()
def log(msg): print(f"[{time.time()-_T0:8.1f}s] {msg}", flush=True)


@contextlib.contextmanager
def _suppress_stdout():
    import io
    dn, old = io.StringIO(), sys.stdout
    try: sys.stdout = dn; yield
    finally: sys.stdout = old


# ---- broad offline-asset discovery (same approach as the inference v4) -----
def _find_under_input(filename):
    root = Path("/kaggle/input")
    if not root.exists(): return None
    try:
        for p in root.rglob(filename): return p
    except Exception: pass
    return None

def _find_wheels_dirs():
    root = Path("/kaggle/input")
    if not root.exists(): return []
    found = []
    try:
        for d in root.rglob("wheels"):
            if d.is_dir() and any(d.glob("*.whl")): found.append(str(d))
    except Exception: pass
    return list(dict.fromkeys(found))

def _find_torch_hub_dir():
    # Must hold the LPIPS(alexnet)/DISTS(vgg16) torchvision backbone weights -- NOT
    # just any "hub/checkpoints" dir. The train-assets dataset also leaves behind a
    # "hub/checkpoints/dinov2_vitb14_pretrain.pth" cache from its DINOv2 staging step
    # (lovif-stage-train-assets.py's torch.hub.load side effect); matching on a bare
    # "*.pth" would risk picking THAT dir and pointing TORCH_HOME at the wrong cache,
    # silently breaking LPIPS/DISTS backbone init offline.
    root = Path("/kaggle/input")
    if not root.exists(): return None
    want = ("alexnet", "vgg")
    try:
        for ckpt_dir in root.rglob("checkpoints"):
            if not (ckpt_dir.is_dir() and ckpt_dir.parent.name == "hub"):
                continue
            names = [p.name.lower() for p in ckpt_dir.glob("*.pth")]
            if any(any(w in n for w in want) for n in names):
                return ckpt_dir.parent.parent
    except Exception: pass
    return None

def _find_train_dir():
    root = Path("/kaggle/input")
    if not root.exists(): return None
    for d in root.rglob("dataset_train"):
        if d.is_dir() and any(f.suffix.lower() == ".png" for f in list(d.iterdir())[:50] if f.is_file()):
            return d
    for d in root.rglob("*"):
        if d.is_dir() and "val" not in d.name.lower() and "test" not in d.name.lower():
            pngs = [f for f in list(d.iterdir())[:64] if f.is_file() and f.suffix.lower() == ".png"]
            if len(pngs) >= 32: return d
    return None

def _find_val_dir():
    root = Path("/kaggle/input")
    if not root.exists(): return None
    for d in root.rglob("dataset_val"):
        if d.is_dir() and any(f.suffix.lower() == ".png" for f in d.iterdir() if f.is_file()):
            return d
    return None

def _find_train_assets():
    """Locate the GAN/CLIP backbones for the full adversarial loss. Prefers DINOv3 via
       plain HF Transformers (AutoModel.from_pretrained on a staged snapshot dir — the
       OFFICIAL, tested implementation; NOT a hand-converted state_dict, see the
       training-pivot memory note on why the manual native-format conversion was
       abandoned: verified output mismatch). Falls back to DINOv2 (ungated, staged by
       lovif-stage-train-assets.py). Returns a dict:
         dinov3_snapshot: dir with config.json + model.safetensors (HF snapshot)
         dinov2         : the staged DINOv2 dir (hub repo + weight) — fallback
         clip           : staged clip-vit-base-patch32 dir for the CLIP loss"""
    root = Path("/kaggle/input")
    res = {"dinov3_snapshot": None, "dinov2": None, "clip": None}
    if not root.exists(): return res
    try:
        # DINOv3 HF snapshot: a dir with config.json (model_type mentions dinov3) + a weight file
        for cfg in root.rglob("config.json"):
            try:
                txt = cfg.read_text(encoding="utf-8", errors="ignore").lower()
            except Exception:
                continue
            if "dinov3" not in txt:
                continue
            d = cfg.parent
            if (d / "model.safetensors").exists() or (d / "pytorch_model.bin").exists():
                res["dinov3_snapshot"] = d; break
        # DINOv2 fallback (lovif-stage-train-assets.py layout)
        for d in root.rglob("dinov2"):
            if d.is_dir() and any(d.rglob("*.pth")) and any(d.rglob("hubconf.py")):
                res["dinov2"] = d; break
        for d in root.rglob("clip-vit-base-patch32"):
            if d.is_dir() and any(d.glob("*.json")): res["clip"] = d; break
    except Exception: pass
    return res


def _stub_torch_geometric():
    """compressai 1.2.8's TOP-LEVEL __init__.py EAGERLY imports its point-cloud
    machinery (verified against the 1.2.8 source): `import compressai` pulls in
    compressai.transforms -> `from .point import *`, and compressai.datasets/losses
    -> pointcloud, which contain UNGUARDED imports of torch_geometric:
        compressai/registry/transforms.py : import torch_geometric.transforms
        compressai/transforms/point/*.py  : from torch_geometric.data import Data
                                            from torch_geometric.data.datapipes import functional_transform
                                            from torch_geometric.transforms import BaseTransform, Center
      and those files use `@functional_transform("name")` as a CLASS DECORATOR and
      subclass `BaseTransform`. This project only uses compressai.entropy_models
      (EntropyBottleneck/GaussianConditional) for 2D image coding -- none of the
      point-cloud code is ever executed -- but the import must still succeed.
      (AEIC_trainer.py's `from codec.codec_trainer import PixelCodec_ME_Trainer` is
      what first triggers `import compressai`; the inference-only v4 script imports
      AEIC_practical, not AEIC_trainer, which is why this never surfaced before.)

    torch_geometric is a heavy package whose accelerated extensions
    (torch-scatter/sparse/cluster) must match the exact torch+CUDA build -- the kind
    of fragile dependency that has no place in an offline pipeline that never uses it.
    So if it isn't already importable, we install a META-PATH FINDER that fabricates a
    stub module for ANY `torch_geometric[.*]` import. Each stub's attributes resolve to
    a universal placeholder class that works in all three ways compressai uses them:
      (1) as a base class      -> `class X(BaseTransform): ...`  (it's a real class)
      (2) as a decorator       -> `@functional_transform("n")` returns the class
                                   unchanged (placeholder() -> instance; instance(cls)
                                   -> cls)
      (3) as a constructor/type-> `Data(...)` / `data: Data` annotations
    Other optional compressai deps (range_coder, pyntcloud, h5py, pointops, open3d) are
    all try/except-guarded in 1.2.8, and pytorch_msssim/einops/tqdm are installed/in
    the base image, so torch_geometric is the only unguarded gap. Safe: the placeholder
    code is never actually invoked by our entropy_models-only usage."""
    if importlib.util.find_spec("torch_geometric") is not None:
        return  # a real install is already present -- nothing to stub

    class _TGAny:
        """Universal placeholder: subclassable, constructible, and usable as a
        decorator factory (functional_transform('n')(cls) -> cls)."""
        def __init__(self, *a, **k):
            pass
        def __call__(self, *a, **k):
            if len(a) == 1 and not k and (isinstance(a[0], type) or callable(a[0])):
                return a[0]          # passthrough decorator: return the class/fn
            return self

    _name_cache = {}
    def _module_getattr(name):
        # dunders must raise AttributeError so the stub behaves like a real module
        # with no __file__ (see the earlier torchvision/inspect crash for why).
        if name.startswith("__") and name.endswith("__"):
            raise AttributeError(name)
        if name not in _name_cache:   # distinct subclass per name -> no duplicate-base errors
            _name_cache[name] = type(f"_TG_{name}", (_TGAny,), {})
        return _name_cache[name]

    class _TGFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        def find_spec(self, fullname, path=None, target=None):
            if fullname == "torch_geometric" or fullname.startswith("torch_geometric."):
                return importlib.machinery.ModuleSpec(fullname, self)
            return None
        def create_module(self, spec):
            m = types.ModuleType(spec.name)
            m.__getattr__ = _module_getattr   # PEP 562 module-level __getattr__
            m.__path__ = []                    # mark as package so submodule imports recurse here
            return m
        def exec_module(self, module):
            pass

    if not any(type(f).__name__ == "_TGFinder" for f in sys.meta_path):
        sys.meta_path.append(_TGFinder())     # appended LAST: only fires when the real
                                              # torch_geometric is genuinely absent
    log("  installed torch_geometric stub finder (compressai's unused point-cloud deps)")


# =============================================================================
#  STEP 1 — offline environment (no network when staged datasets exist)
# =============================================================================
def setup_env():
    log("STEP 1 — environment setup (offline-first)")
    _stub_torch_geometric()
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
    work.mkdir(parents=True, exist_ok=True)
    is_offline = ASSUME_OFFLINE
    if is_offline:
        os.environ.setdefault("HF_HUB_OFFLINE", "1")
        os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
        os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

    def pip(*a):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

    repo = None
    hit = _find_under_input("AEIC_practical.py")
    if hit is not None:
        repo = hit.parent.parent if hit.parent.name == "src" else hit.parent
    log(f"  AEIC source: {'OK' if repo else 'MISSING'} -> {repo}")

    if is_offline:
        wheels = _find_wheels_dirs()
        if wheels:
            log(f"  OFFLINE wheelhouse(s): {wheels}")
            fl = []
            for w in wheels: fl += [f"--find-links={w}"]
            pip("--no-index", *fl, "--no-deps",
                "tokenizers==0.20.3", "pytorch-msssim", "compressai", "lpips", "DISTS_pytorch",
                "diffusers==0.25.1", "huggingface_hub==0.25.0", "peft",
                "transformers==4.46.3", "einops", "timm", "safetensors",
                "accelerate", "omegaconf", "constriction")
        else:
            log("  OFFLINE: no wheelhouse found — deps may be MISSING. Stage wheels/ and attach it.")
    else:
        pip("--no-deps", "compressai"); pip("lpips", "DISTS_pytorch", "pytorch-msssim")
        pip("diffusers==0.25.1", "huggingface_hub==0.25.0", "peft", "transformers==4.46.3",
            "einops", "timm", "safetensors", "accelerate", "omegaconf", "constriction")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

    hub_root = _find_torch_hub_dir()
    if hub_root is not None:
        os.environ["TORCH_HOME"] = str(hub_root)
        log(f"  TORCH_HOME -> {hub_root} (staged LPIPS/DISTS backbone weights)")
    elif is_offline:
        log("  !! no staged torch.hub cache — LPIPS/DISTS backbone init will fail offline. "
            "Run lovif-stage-metric-backbones.py and attach it.")

    miss = [m for m in ("lpips", "DISTS_pytorch", "compressai") if importlib.util.find_spec(m) is None]
    if miss:
        raise RuntimeError(f"Required modules missing after setup: {miss}. Attach the wheels/ dataset.")
    try:
        importlib.invalidate_caches()
        import transformers, tokenizers, diffusers, peft  # noqa: F401
    except Exception as e:
        raise RuntimeError(
            "HF stack failed to import after offline install — a transitive dep is at an "
            f"incompatible version.\n    {type(e).__name__}: {e}\nAdd that exact package==version "
            "to the staged wheels and re-run.")
    if repo is None:
        raise RuntimeError("AEIC source not found under /kaggle/input. Attach the AEIC staging dataset.")
    return work, Path(repo), is_offline


# =============================================================================
#  STEP 2 — locate SD-Turbo + halfDecoder + warm-start AEIC_ME .pkl (offline)
# =============================================================================
def setup_models(work):
    log("STEP 2 — models & warm-start checkpoint")
    sd_dir = None
    hit = _find_under_input("model_index.json")
    if hit is not None and (hit.parent / "unet").exists() and (hit.parent / "vae").exists():
        sd_dir = hit.parent
    vae_dec = _find_under_input("halfDecoder.ckpt")

    def ftnum(p):
        m = re.search(r"ft(\d+)", Path(p).name); return int(m.group(1)) if m else 0
    hits, seen = [], set()
    root = Path("/kaggle/input")
    if root.exists():
        for p in root.rglob("*.pkl"):
            n = p.name
            if "pretrain" in n.lower() or "ME" not in n: continue
            if n not in seen: hits.append(p); seen.add(n)
    by_stem = {p.stem: p for p in sorted(hits, key=ftnum, reverse=True)}
    warm = None
    for cand in WARMSTART_NAMES:
        if cand in by_stem: warm = by_stem[cand]; break
    if warm is None and by_stem: warm = list(by_stem.values())[0]

    log(f"  sd-turbo:    {'OK' if sd_dir and (sd_dir/'model_index.json').exists() else 'MISSING'} {sd_dir}")
    log(f"  halfDecoder: {'OK' if vae_dec and Path(vae_dec).exists() else 'MISSING'} {vae_dec}")
    log(f"  warm-start:  {'OK' if warm else 'MISSING'} {warm}  (candidates: {list(by_stem)})")
    missing = []
    if not (sd_dir and (sd_dir / "model_index.json").exists()): missing.append("sd-turbo snapshot")
    if not (vae_dec and Path(vae_dec).exists()): missing.append("halfDecoder.ckpt")
    if warm is None: missing.append("AEIC_ME_*.pkl warm-start checkpoint")
    if missing:
        raise RuntimeError("OFFLINE STAGING INCOMPLETE — missing:\n  - " + "\n  - ".join(missing) +
                           "\nAttach the AEIC staging dataset + your checkpoint dataset.")
    return sd_dir, Path(vae_dec), Path(warm), by_stem


# =============================================================================
#  Datasets
# =============================================================================
def build_train_loader(train_dir, subset_override=None, patch_override=None, batch_override=None):
    import torch
    import numpy as np
    from PIL import Image
    import torchvision.transforms as T
    from torch.utils.data import Dataset, DataLoader
    Image.MAX_IMAGE_PIXELS = None

    crop  = patch_override if patch_override is not None else PATCH
    batch = batch_override if batch_override is not None else BATCH

    files = sorted(p for p in Path(train_dir).rglob("*.png"))
    if not files:
        files = sorted(p for p in Path(train_dir).rglob("*.*")
                       if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))
    n_take = subset_override if subset_override is not None else TRAIN_SUBSET
    if n_take: files = files[:n_take]
    log(f"  train images: {len(files)} from {train_dir} (crop={crop}, batch={batch})")
    if not files: raise RuntimeError(f"no training images under {train_dir}")
    norm = T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

    class DS(Dataset):
        def __init__(self, files): self.files = files
        def __len__(self): return len(self.files)
        def __getitem__(self, i):
            for _ in range(8):
                try: im = Image.open(self.files[i]).convert("RGB"); break
                except Exception: i = random.randrange(len(self.files))
            else: im = Image.new("RGB", (crop, crop))
            w, h = im.size
            if w < crop or h < crop:
                s = max(crop / w, crop / h)
                im = im.resize((max(crop, int(w * s) + 1), max(crop, int(h * s) + 1)), Image.LANCZOS)
                w, h = im.size
            x = random.randint(0, w - crop); y = random.randint(0, h - crop)
            im = im.crop((x, y, x + crop, y + crop))
            if random.random() < 0.5: im = im.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5: im = im.transpose(Image.FLIP_TOP_BOTTOM)
            t = torch.from_numpy(np.asarray(im, np.float32).transpose(2, 0, 1) / 255.0)
            return norm(t)

    return DataLoader(DS(files), batch_size=batch, shuffle=True, num_workers=4,
                      pin_memory=True, drop_last=True, persistent_workers=True)


def load_val_crops(val_dir, n, device):
    import torch
    import numpy as np
    from PIL import Image
    if val_dir is None: return []
    files = sorted(Path(val_dir).glob("*.png"))[:n]
    out = []
    for p in files:
        im = Image.open(p).convert("RGB"); w, h = im.size
        s = max(PATCH / w, PATCH / h) if (w < PATCH or h < PATCH) else 1.0
        if s != 1.0:
            im = im.resize((max(PATCH, int(w * s) + 1), max(PATCH, int(h * s) + 1)), Image.LANCZOS); w, h = im.size
        x = (w - PATCH) // 2; y = (h - PATCH) // 2
        im = im.crop((x, y, x + PATCH, y + PATCH))
        t = torch.from_numpy(np.asarray(im, np.float32).transpose(2, 0, 1) / 255.0)
        out.append((t * 2 - 1).unsqueeze(0).to(device))
    return out


# =============================================================================
#  Board-aligned OFFLINE loss modules.
#  DISTS_pytorch + lpips(alex) are the SAME backbones the board-faithful scorer
#  uses (already staged for inference). DINOv2 + CLIP are the T4-staged extras.
# =============================================================================
class _Sobel:
    """Edge magnitude (mirrors oc_ea_dists_loss.Sobel), per-image min-max to [0,1]."""
    def __init__(self, device):
        import torch, torch.nn as nn
        self.conv = nn.Conv2d(1, 2, 3, 1, 1, bias=False).to(device)
        Gx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]])
        Gy = torch.tensor([[-1., -2, -1], [0, 0, 0], [1, 2, 1]])
        # Reassigning .weight with a fresh Parameter OVERWRITES the .to(device) done above --
        # the new tensor must be moved to device itself, or the conv silently reverts to CPU.
        self.conv.weight = nn.Parameter(torch.stack([Gx, Gy]).unsqueeze(1).to(device), requires_grad=False)
        self.w = torch.tensor([0.2989, 0.5870, 0.1140], device=device).view(1, 3, 1, 1)
    def __call__(self, x):  # x in [0,1], (B,3,H,W) -> (B,3,H,W) edge in [0,1]
        import torch
        g = (x * self.w).sum(1, keepdim=True)
        e = torch.sqrt((self.conv(g) ** 2).sum(1, keepdim=True) + 1e-6)
        e = e / (e.amax(dim=(2, 3), keepdim=True) + 1e-6)
        return e.repeat(1, 3, 1, 1)


def _OC(x, patch_size=224):
    """Overlap-chunk a square image into patch_size tiles (mirrors oc_ea_dists_loss.OC)."""
    import torch.nn.functional as F
    _, C, H, W = x.shape
    if H == W and H <= patch_size: return x
    if H % patch_size == 0: stride = patch_size
    else:
        N = int(H / patch_size) + 1
        stride = patch_size - (N * patch_size - H) // max(1, (N - 1))
    p = F.unfold(x, kernel_size=patch_size, stride=stride)
    return p.permute(0, 2, 1).reshape(-1, C, patch_size, patch_size)


def charbonnier(a, b, eps=1e-3):
    """Smooth-L1 (Charbonnier) distortion anchor, replacing plain MSE for the loss. All three
    research reports (and SPRDiff's loss design) prefer an L1-family anchor: MSE rewards regressing
    toward the blurry pixel-mean of uncertain texture -- the exact over-smoothing that tanks LPIPS/
    DISTS -- whereas Charbonnier gives a stable gradient without exponentially punishing the small
    pixel deviations inherent to generative texture synthesis. Inputs are in [-1,1] (same as the
    old mse(rec,x)); eval PSNR still uses true MSE separately, so the reported PSNR is unaffected."""
    import torch
    return torch.sqrt((a - b) ** 2 + eps * eps).mean()


def build_losses(device):
    import torch
    import torch.nn.functional as F
    import numpy as np
    import lpips as _l
    import DISTS_pytorch
    from DISTS_pytorch import DISTS as _D
    log("  loading LPIPS(alex) + DISTS(weights.pt) + MS-SSIM for the training loss ...")
    LP = _l.LPIPS(net="alex", verbose=False).to(device).eval()
    #  VGG-dominant TRAINING net (see LPIPS_VGG_SHARE config). Loaded defensively: the VGG16 torch-hub
    #  backbone is already staged (DISTS uses it) and lpips' own lin-weights ship in-package, so this
    #  should load offline -- but if it doesn't for any reason, fall back to Alex-only so a run never
    #  dies for this. LP (alex) is ALWAYS the eval/selection net regardless (= the board metric).
    LP_VGG = None
    if LPIPS_VGG_SHARE > 0:
        try:
            LP_VGG = _l.LPIPS(net="vgg", verbose=False).to(device).eval()
            for p in LP_VGG.parameters(): p.requires_grad_(False)
            log(f"    LPIPS train-blend ON: {LPIPS_VGG_SHARE:.2f}*VGG + {LPIPS_ALEX_SHARE:.2f}*Alex "
                "(eval stays pure Alex = board)")
        except Exception as e:
            LP_VGG = None
            log(f"    !! LPIPS-VGG load failed ({type(e).__name__}: {e}) -> training on Alex-only "
                "(old behavior); eval unaffected.")
    DS = _D(load_weights=False)
    wp = os.path.join(os.path.dirname(DISTS_pytorch.__file__), "weights.pt")
    w = torch.load(wp, map_location="cpu"); DS.alpha.data, DS.beta.data = w["alpha"], w["beta"]
    DS = DS.to(device).eval()
    for p in list(LP.parameters()) + list(DS.parameters()): p.requires_grad_(False)
    sobel = _Sobel(device)

    # ---- differentiable MS-SSIM (board-faithful: RGB->Y, 5-scale, same as scorer) ----
    def _fspecial(size=11, sigma=1.5):
        mm = (size - 1) / 2.0
        y, x = np.ogrid[-mm:mm + 1, -mm:mm + 1]
        h = np.exp(-(x * x + y * y) / (2.0 * sigma * sigma))
        h[h < np.finfo(h.dtype).eps * h.max()] = 0
        if h.sum() != 0: h /= h.sum()
        return torch.from_numpy(h)
    WIN = _fspecial().float().view(1, 1, 11, 11).to(device)
    YW = torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)

    def _to_y255(x01):  # keep differentiable (no rounding) for the loss
        return (x01 * YW).sum(1, keepdim=True) * 255.0

    def _ssim(X, Y, dr=255.0):
        C1, C2 = (0.01 * dr) ** 2, (0.03 * dr) ** 2
        mu1, mu2 = F.conv2d(X, WIN), F.conv2d(Y, WIN)
        m1s, m2s, m12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
        s1 = F.conv2d(X * X, WIN) - m1s; s2 = F.conv2d(Y * Y, WIN) - m2s
        s12 = F.conv2d(X * Y, WIN) - m12
        cs = F.relu((2 * s12 + C2) / (s1 + s2 + C2))
        return (((2 * m12 + C1) / (m1s + m2s + C1)) * cs).mean([1, 2, 3]), cs.mean([1, 2, 3])

    def ms_ssim(xh_pm1, x_pm1):
        X, Y = _to_y255(xh_pm1 * 0.5 + 0.5), _to_y255(x_pm1 * 0.5 + 0.5)
        wts = torch.tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333], device=device)
        mcs, sv = [], None
        for _ in range(5):
            sv, cs = _ssim(X, Y); mcs.append(cs)
            pad = (X.shape[2] % 2, X.shape[3] % 2)
            X, Y = F.avg_pool2d(X, 2, padding=pad), F.avg_pool2d(Y, 2, padding=pad)
        mcs = torch.stack(mcs, 0)
        return (torch.prod(mcs[:-1] ** wts[:-1].unsqueeze(1), 0) * (sv ** wts[-1])).mean()

    def msssim_loss(xh_pm1, x_pm1):
        return 1.0 - ms_ssim(xh_pm1, x_pm1)

    def lpips_loss(xh_pm1, x_pm1):
        #  VGG-dominant blend for OPTIMIZATION (robust gradients, harder to game); falls back to
        #  pure Alex if VGG didn't load. Eval/selection always uses LP (alex) directly elsewhere.
        if LP_VGG is None:
            return LP(xh_pm1, x_pm1).mean()
        return (LPIPS_VGG_SHARE * LP_VGG(xh_pm1, x_pm1).mean()
                + LPIPS_ALEX_SHARE * LP(xh_pm1, x_pm1).mean())

    def dists_oc_ea(xh_pm1, x_pm1):
        """OC-EA-DISTS: 1.5*DISTS(OC(img)) + 0.5*DISTS(OC(edge)), in [0,1]."""
        a = (xh_pm1 * 0.5 + 0.5).clamp(0, 1); b = (x_pm1 * 0.5 + 0.5).clamp(0, 1)
        d = DS(_OC(a), _OC(b)).mean()
        e = DS(_OC(sobel(a)), _OC(sobel(b))).mean()
        return 1.5 * d + 0.5 * e

    return lpips_loss, dists_oc_ea, msssim_loss, LP, DS


def build_clip_loss(clip_dir, device):
    """CLIP-cos loss (mirrors training_utils.CLIPLoss) from a STAGED HF snapshot."""
    import torch
    import torchvision.transforms as T
    from transformers import CLIPVisionModelWithProjection
    enc = CLIPVisionModelWithProjection.from_pretrained(str(clip_dir), local_files_only=True).to(device).eval()
    enc.requires_grad_(False)
    tf = T.Compose([T.Resize(224), T.CenterCrop(224),
                    T.Normalize([0.48145466, 0.4578275, 0.40821073],
                                [0.26862954, 0.26130258, 0.27577711])])
    def clip_loss(xh_pm1, x_pm1):
        a = tf((xh_pm1 * 0.5 + 0.5)); b = tf((x_pm1 * 0.5 + 0.5))
        fa = enc(a).image_embeds; fb = enc(b).image_embeds
        fa = fa / fa.norm(p=2, dim=-1, keepdim=True); fb = fb / fb.norm(p=2, dim=-1, keepdim=True)
        return torch.norm(fb - fa, p=2, dim=-1).mean()
    return clip_loss


def _load_dinov3_hf(snapshot_dir, device):
    """Load DINOv3-ViT-B/16 via plain HF Transformers (AutoModel) from a LOCAL staged
       snapshot (config.json + model.safetensors). This is the OFFICIAL, tested
       implementation -- NOT a hand-converted state_dict. (A manual conversion from the
       HF format into the cloned dinov3/ repo's native key layout was attempted and
       REJECTED: a forward-pass parity check showed the converted model's outputs
       diverge sharply from HF's own outputs -- max abs CLS diff ~4.8, i.e. genuinely
       different features, most likely from undocumented RoPE/token-layout differences
       between the two independently-implemented codebases. Rather than debug a fragile
       bit-for-bit port, we just use HF's verified-correct model directly.)
       Returns (model, n_special_tokens) -- n_special_tokens = 1 (CLS) + num register
       tokens, read from the checkpoint's own register_tokens shape (no hardcoding)."""
    from transformers import AutoModel
    model = AutoModel.from_pretrained(str(snapshot_dir), local_files_only=True)
    model = model.to(device).eval().requires_grad_(False)
    sd = model.state_dict()
    reg_key = next((k for k in sd if k.endswith("register_tokens")), None)
    n_register = sd[reg_key].shape[1] if reg_key is not None else 0
    return model, 1 + n_register   # +1 for the CLS token


def _load_dinov2(dinov2_dir, device):
    import torch
    repo_dir = None
    for h in Path(dinov2_dir).rglob("hubconf.py"):
        repo_dir = h.parent; break
    wpath = None
    for w in Path(dinov2_dir).rglob("*.pth"):
        if "vitb14" in w.name or "vit_b" in w.name or "vitb" in w.name: wpath = w; break
    if wpath is None:
        for w in Path(dinov2_dir).rglob("*.pth"): wpath = w; break
    if repo_dir is None or wpath is None:
        raise RuntimeError(f"DINOv2 repo/weight not found under {dinov2_dir}")
    backbone = torch.hub.load(str(repo_dir), "dinov2_vitb14", source="local", pretrained=False)
    sd = torch.load(str(wpath), map_location="cpu")
    backbone.load_state_dict(sd, strict=False)
    return backbone.to(device).eval().requires_grad_(False)


def build_vision_aided_gan(assets, device):
    """Vision-aided hinge GAN: FROZEN ViT features (DINOv3 preferred, else DINOv2) +
       trainable conv head. Same mechanism as vision_aided_loss. DINOv3 is the recipe's
       actual backbone (gated weights); DINOv2 is the ungated fallback. Returns
       (G_loss(rec), D_step(rec,real), D_params) or None on failure."""
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.nn.utils import spectral_norm as SN
    backbone = None
    backend = None       # "hf" (DINOv3 via Transformers) or "hub" (DINOv2 via torch.hub)
    n_special = 0
    if assets.get("dinov3_snapshot") is not None:
        try:
            backbone, n_special = _load_dinov3_hf(assets["dinov3_snapshot"], device)
            backend = "hf"
            log(f"  GAN: frozen DINOv3-ViT-B/16 backbone loaded via HF Transformers "
                f"({Path(assets['dinov3_snapshot']).name}, {n_special} special tokens)")
        except Exception as e:
            log(f"  GAN: DINOv3 (HF) load failed ({type(e).__name__}: {e})")
            log("       NOTE: AEIC pins transformers==4.46.3 (for its diffusers-0.25 stack), which "
                "PREDATES DINOv3 support (added ~transformers 4.53). So the DINOv3 HF snapshot CANNOT "
                "load on this offline box. Use DINOv2 instead: run lovif-stage-train-assets.py on a T4 "
                "and attach its output. Falling back to DINOv2 if staged ...")
            backbone = None
    if backbone is None and assets.get("dinov2") is not None:
        try:
            backbone = _load_dinov2(assets["dinov2"], device)
            backend = "hub"
            log(f"  GAN: frozen DINOv2-ViT-B/14 backbone loaded ({Path(assets['dinov2']).name})")
        except Exception as e:
            log(f"  GAN: DINOv2 load failed ({type(e).__name__}: {e}) -> adversarial term OFF")
            backbone = None
    if backbone is None:
        return None

    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    def feats(x_pm1):
        x = F.interpolate(x_pm1 * 0.5 + 0.5, size=(224, 224), mode="area")
        x = (x - mean) / std
        if backend == "hf":
            # last_hidden_state: (B, n_special + n_patches, 768); drop CLS+register
            # tokens, reshape the remaining patch tokens to a spatial (B,768,h,w) map.
            out = backbone(pixel_values=x).last_hidden_state
            patches = out[:, n_special:, :]
            B, N, C = patches.shape
            h = w = int(round(N ** 0.5))
            f = patches.permute(0, 2, 1).reshape(B, C, h, w).float()
        else:
            f = backbone.get_intermediate_layers(x, n=1, reshape=True)[0].float()  # (B,768,h,w)
        return f

    head = nn.Sequential(
        SN(nn.Conv2d(768, 256, 1)), nn.LeakyReLU(0.2, True),
        SN(nn.Conv2d(256, 256, 3, 1, 1)), nn.LeakyReLU(0.2, True),
        SN(nn.Conv2d(256, 1, 1)),
    ).to(device)

    def logits(x_pm1, detach_backbone_input=False):
        xin = x_pm1.detach() if detach_backbone_input else x_pm1
        return head(feats(xin))

    def g_loss(rec_pm1):                      # generator wants high logits
        return -head(feats(rec_pm1)).mean()
    def d_loss(rec_pm1, real_pm1):            # hinge discriminator
        lr = head(feats(real_pm1.detach())); lf = head(feats(rec_pm1.detach()))
        return torch.relu(1.0 - lr).mean() + torch.relu(1.0 + lf).mean()

    return g_loss, d_loss, list(head.parameters())


# =============================================================================
#  Trainer construction (shared by the full run and each sweep trial). Each call builds a
#  FRESH AEIC_ME_Trainer warm-started from `warm` -- sweep trials must NOT be cumulative
#  (each lambda_rate candidate needs to start from the same ft16 baseline to be comparable).
# =============================================================================
def build_trainer(repo, sd_dir, vae_dec, warm, device, label="STEP 3"):
    import torch
    sys.path.insert(0, str(repo)); sys.path.insert(0, str(repo / "src"))
    from AEIC_trainer import AEIC_ME_Trainer
    # codec_path is deliberately None: AEIC_ME_Trainer.load_AEIC_state_dict is STRICT and
    # assumes the fresh codec was built with the SAME compressai that SAVED the checkpoint.
    # The public AEIC .pkl files were saved under compressai 1.2.6 (== AEIC's vendored
    # base_entropy_coder.py: EntropyBottleneck registers params _matrix{i}/_bias{i}/_factor{i}),
    # but the staged wheel is 1.2.8 (nn.ParameterList -> matrices.{i}/biases.{i}/factors.{i}),
    # so the strict loader KeyErrors on entropy_bottleneck.matrices.0. We build a random codec
    # and warm-start via _apply_ckpt(), which bridges that pure rename. The 1.2.6 and 1.2.8
    # EntropyBottleneck forward math is byte-identical (verified) -> score-neutral.
    args = types.SimpleNamespace(
        sd_path=str(sd_dir), codec_path=None, vae_decoder_path=str(vae_dec),
        lora_rank_unet=32, compile_model=False,
        use_tiled_unet=False, use_tiled_vae=False,
        enable_xformers_memory_efficient_attention=False,
        latent_tiled_size=96, latent_tiled_overlap=32, vae_decoder_tiled_size=160)
    log(f"{label} — building AEIC-ME trainer (warm-start) ...")
    with _suppress_stdout():
        net = AEIC_ME_Trainer(sd_path=str(sd_dir), args=args).to(device)
    _apply_ckpt(net, torch.load(str(warm), map_location="cpu"), what="warm-start")
    trainable = [p for p in net.parameters() if p.requires_grad]
    log(f"  trainable params: {sum(p.numel() for p in trainable)/1e6:.2f}M")
    return net, trainable


# =============================================================================
#  NATIVE CHECKPOINT CHECK (see CKPT_CHECK_MODE config above): zero training steps, just load
#  each candidate public checkpoint and evaluate it as-is, to find the best warm-start point.
# =============================================================================
def run_ckpt_check(repo, sd_dir, vae_dec, ckpt_candidates, val_dir, device):
    import torch
    LP_DS = build_losses(device)   # (lpips_loss, dists_loss, msssim_loss, LP, DS) -- only need LP/DS here
    LP, DS = LP_DS[3], LP_DS[4]
    mse = torch.nn.MSELoss()
    val_crops = load_val_crops(val_dir, CKPT_CHECK_EVAL_N, device)
    if not val_crops:
        raise RuntimeError("CKPT_CHECK_MODE needs dataset_val crops -- attach/verify dataset_val.")

    found = {n: ckpt_candidates[n] for n in CKPT_CHECK_NAMES if n in ckpt_candidates}
    missing = [n for n in CKPT_CHECK_NAMES if n not in ckpt_candidates]
    if missing:
        log(f"  !! not staged, skipping: {missing}  (available: {list(ckpt_candidates)})")
    if not found:
        raise RuntimeError(f"none of CKPT_CHECK_NAMES={CKPT_CHECK_NAMES} were found among "
                           f"staged checkpoints: {list(ckpt_candidates)}")

    log(f"STEP 4 — NATIVE CHECKPOINT CHECK: {list(found)} (0 training steps, eval_n={CKPT_CHECK_EVAL_N})")
    results = []
    for name, path in found.items():
        log("-" * 70)
        net, _ = build_trainer(repo, sd_dir, vae_dec, path, device, label=f"  [native check {name}]")
        net.eval()
        accs = {k: 0.0 for k in ("bpp", "psnr", "lpips", "dists")}
        with torch.no_grad():
            for x in val_crops:
                x_hat, RLO, res = net.codec(x, ori_h=PATCH, ori_w=PATCH)
                rec = net.vae.decoder(net.unet(x_hat) + res).clamp(-1, 1)
                x01, r01 = (x * 0.5 + 0.5), (rec * 0.5 + 0.5)
                accs["bpp"]   += float(RLO.quantized_total_bpp)
                accs["psnr"]  += float(10 * torch.log10(1.0 / mse(r01, x01)))
                accs["lpips"] += float(LP(rec, x).mean())
                accs["dists"] += float(DS(r01.clamp(0, 1), x01.clamp(0, 1)).mean())
        n = len(val_crops); a = {k: v / n for k, v in accs.items()}
        log(f"  [native RESULT] {name:16s}  bpp={a['bpp']:.5f}  PSNR={a['psnr']:.2f}  "
            f"LPIPS={a['lpips']:.4f}  DISTS={a['dists']:.4f}")
        results.append((name, a))
        del net; gc.collect(); torch.cuda.empty_cache()

    log("=" * 70)
    log("  NATIVE CHECKPOINT SUMMARY (target bpp <= 0.008; internal target ~0.0075)")
    log(f"  {'checkpoint':>16s}  {'bpp':>8s}  {'PSNR':>6s}  {'LPIPS':>7s}  {'DISTS':>7s}")
    for name, a in results:
        flag = "  <-- under 0.008" if a["bpp"] <= 0.008 else ""
        log(f"  {name:16s}  {a['bpp']:8.5f}  {a['psnr']:6.2f}  {a['lpips']:7.4f}  {a['dists']:7.4f}{flag}")
    log("  Pick whichever sits closest to budget as WARMSTART_NAMES[0] for the lambda sweep "
        "(it'll need the smallest external lambda_rate push to reach our target).")
    log("=" * 70)
    return 0


# =============================================================================
#  RATE SWEEP (see the 2026-06-30 finding above LAMBDA_RATE): probes several lambda_rate
#  values x several warm-start checkpoints with SHORT trials (each fresh, not cumulative) to
#  find where eval bpp/quality actually lands, before committing a full ~30min run.
#  Loops over BOTH axes (checkpoint x lambda) at the SAME training depth so the comparison is
#  fair: the native check showed ft32 beating "ft16 forced down via 300 quick steps" on DISTS,
#  but that compared ft32's ZERO-training native numbers to a possibly-undertrained ft16 -- not
#  a fair test of whether ft16 needs more steps to reach the same place. Testing both checkpoints
#  here, at the now-bumped SWEEP_STEPS=600, settles that properly instead of guessing.
# =============================================================================
def run_sweep(repo, sd_dir, vae_dec, ckpt_candidates, train_dir, val_dir, assets, clip_dir, device):
    import torch
    lpips_loss, dists_loss, msssim_loss, LP, DS = build_losses(device)
    mse = torch.nn.MSELoss()
    clip_loss = None
    if clip_dir is not None and LAMBDA_CLIP > 0:
        try:
            clip_loss = build_clip_loss(clip_dir, device); log(f"  CLIP loss ON (staged {clip_dir.name})")
        except Exception as e:
            log(f"  CLIP loss OFF (staged load failed: {e})")
    gan = None
    if LAMBDA_ADV > 0 and (assets["dinov3_snapshot"] is not None or assets["dinov2"] is not None):
        gan = build_vision_aided_gan(assets, device)
    g_loss = d_loss = None; disc_params = []
    if gan is not None:
        g_loss, d_loss, disc_params = gan

    val_crops = load_val_crops(val_dir, SWEEP_EVAL_N, device)
    if not val_crops:
        raise RuntimeError("SWEEP_MODE needs dataset_val crops to compare candidates -- "
                           "attach/verify the LoViF dataset_val.")
    train_loader = build_train_loader(train_dir, subset_override=SWEEP_TRAIN_SUBSET)

    cands = {n: ckpt_candidates[n] for n in SWEEP_WARMSTARTS if n in ckpt_candidates}
    missing = [n for n in SWEEP_WARMSTARTS if n not in ckpt_candidates]
    if missing:
        log(f"  !! not staged, skipping: {missing}  (available: {list(ckpt_candidates)})")
    if not cands:
        raise RuntimeError(f"none of SWEEP_WARMSTARTS={SWEEP_WARMSTARTS} were found among "
                           f"staged checkpoints: {list(ckpt_candidates)}")

    log(f"STEP 4 — RATE SWEEP: {list(cands)} x {SWEEP_LAMBDAS} x {SWEEP_STEPS} steps each "
        f"(subset={SWEEP_TRAIN_SUBSET}, eval_n={SWEEP_EVAL_N})")
    results = []
    for ck_name, ck_path in cands.items():
        for lam in SWEEP_LAMBDAS:
            log("-" * 70)
            net, trainable = build_trainer(repo, sd_dir, vae_dec, ck_path, device,
                                           label=f"  [sweep {ck_name} lambda={lam}]")
            opt = torch.optim.AdamW(trainable, lr=LR)
            Dopt = torch.optim.AdamW(disc_params, lr=LR_DISC) if (gan is not None and disc_params) else None

            net.codec.train()
            t_last = time.time()
            for step, batch in enumerate(_cycle(train_loader), start=1):
                if step > SWEEP_STEPS: break
                x = batch.to(device, non_blocking=True).float()
                x_hat, RLO, res = net.codec(x, ori_h=PATCH, ori_w=PATCH)
                rec = net.vae.decoder(net.unet(x_hat) + res).clamp(-1, 1)
                l_rate, l_l2 = RLO.rate_loss, charbonnier(rec, x)   # Charbonnier anchor (was MSE)
                l_msssim, l_dists, l_lpips = msssim_loss(rec, x), dists_loss(rec, x), lpips_loss(rec, x)
                loss = (lam * l_rate + LAMBDA_L2 * l_l2 + LAMBDA_MSSSIM * l_msssim
                        + LAMBDA_DISTS * l_dists + LAMBDA_LPIPS * l_lpips)
                if clip_loss is not None:
                    loss = loss + LAMBDA_CLIP * clip_loss(rec, x)
                aw = LAMBDA_ADV * min(1.0, step / max(1, GAN_WARMUP)) if gan is not None else 0.0
                if gan is not None and aw > 0:
                    try:
                        loss = loss + aw * g_loss(rec)
                    except Exception:
                        pass  # adversarial term is secondary; never let it break a sweep trial
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
                opt.step()
                if gan is not None and aw > 0 and Dopt is not None:
                    try:
                        Dopt.zero_grad(set_to_none=True)
                        dl = d_loss(rec, x); dl.backward()
                        torch.nn.utils.clip_grad_norm_(disc_params, GRAD_CLIP)
                        Dopt.step()
                    except Exception:
                        pass
                if step % SWEEP_LOG_EVERY == 0:
                    dt = (time.time() - t_last) / SWEEP_LOG_EVERY; t_last = time.time()
                    log(f"    step {step:4d}/{SWEEP_STEPS}  loss={float(loss):.4f}  "
                        f"rate(bpp)={float(l_rate):.5f}  {dt:.2f}s/it")

            net.eval()
            accs = {k: 0.0 for k in ("bpp", "psnr", "lpips", "dists")}
            with torch.no_grad():
                for x in val_crops:
                    x_hat, RLO, res = net.codec(x, ori_h=PATCH, ori_w=PATCH)
                    rec = net.vae.decoder(net.unet(x_hat) + res).clamp(-1, 1)
                    x01, r01 = (x * 0.5 + 0.5), (rec * 0.5 + 0.5)
                    accs["bpp"]   += float(RLO.quantized_total_bpp)
                    accs["psnr"]  += float(10 * torch.log10(1.0 / mse(r01, x01)))
                    accs["lpips"] += float(LP(rec, x).mean())
                    accs["dists"] += float(DS(r01.clamp(0, 1), x01.clamp(0, 1)).mean())
            n = len(val_crops); a = {k: v / n for k, v in accs.items()}
            log(f"  [sweep RESULT] {ck_name:14s} lambda_rate={lam:6.1f}  bpp={a['bpp']:.5f}  "
                f"PSNR={a['psnr']:.2f}  LPIPS={a['lpips']:.4f}  DISTS={a['dists']:.4f}")
            results.append((ck_name, lam, a))
            del net, opt, Dopt
            gc.collect(); torch.cuda.empty_cache()

    # closest-to-budget-from-below = uses the MOST of the 0.008 allowance without exceeding it.
    # Flagging this (not just "any under 0.008") because undershooting wastes bits/quality --
    # the goal is using the full budget, not conservatively sitting well below it.
    under = [r for r in results if r[2]["bpp"] <= 0.008]
    best = max(under, key=lambda r: r[2]["bpp"]) if under else None

    log("=" * 70)
    log("  RATE SWEEP SUMMARY (goal: bpp as CLOSE TO 0.008 as possible without exceeding it --"
        " maximize budget usage, don't undershoot)")
    log(f"  {'warmstart':>14s}  {'lambda_rate':>11s}  {'bpp':>8s}  {'PSNR':>6s}  {'LPIPS':>7s}  {'DISTS':>7s}")
    for ck_name, lam, a in results:
        flag = ""
        if best is not None and (ck_name, lam, a) == best:
            flag = "  <-- closest to budget (uses it best)"
        elif a["bpp"] <= 0.008:
            flag = "  <-- under 0.008"
        log(f"  {ck_name:14s}  {lam:11.1f}  {a['bpp']:8.5f}  {a['psnr']:6.2f}  {a['lpips']:7.4f}  {a['dists']:7.4f}{flag}")
    if best is None:
        log("  !! none of these landed under 0.008 -- try higher lambda_rate values.")
    log("  Standalone (Strategy A): pick the flagged closest-to-budget candidate, set "
        "WARMSTART_NAMES/LAMBDA_RATE to it, SWEEP_MODE=False, run the full job.")
    log("  Mixed via knapsack (Strategy B, often better): a candidate landing a bit ABOVE 0.008 "
        "with clearly better LPIPS/DISTS is fine -- stablecodec's knapsack adaptively downgrades "
        "the cheapest-to-compress images to SE/ft16 so the GLOBAL average still hits 0.008, while "
        "most images keep this higher quality. That per-image adaptivity uses the budget better "
        "than any single uniform-rate checkpoint can.")
    log("=" * 70)
    return 0


def _cycle(loader):
    while True:
        for b in loader: yield b


# =============================================================================
#  MAIN
# =============================================================================
def main():
    import torch
    work, repo, is_offline = setup_env()
    sd_dir, vae_dec, warm, ckpt_candidates = setup_models(work)
    train_dir = _find_train_dir()
    if train_dir is None:
        raise RuntimeError("dataset_train not found under /kaggle/input — attach the LoViF "
                           "training dataset (14,136 imgs).")
    val_dir = _find_val_dir()
    assets = _find_train_assets()
    clip_dir = assets["clip"]
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device != "cuda":
        raise RuntimeError("AEIC finetune requires a CUDA GPU.")

    if CKPT_CHECK_MODE:
        return run_ckpt_check(repo, sd_dir, vae_dec, ckpt_candidates, val_dir, device)
    if SWEEP_MODE:
        return run_sweep(repo, sd_dir, vae_dec, ckpt_candidates, train_dir, val_dir, assets, clip_dir, device)

    # ---- build the lambda-INDEPENDENT pieces ONCE, share across both full runs ----
    lpips_loss, dists_loss, msssim_loss, LP, DS = build_losses(device)
    mse = torch.nn.MSELoss()
    clip_loss = None
    if clip_dir is not None and LAMBDA_CLIP > 0:
        try:
            clip_loss = build_clip_loss(clip_dir, device); log(f"  CLIP loss ON (staged {clip_dir.name})")
        except Exception as e:
            log(f"  CLIP loss OFF (staged load failed: {e})")
    else:
        log("  CLIP loss OFF (no staged CLIP ViT-B/32; optional).")
    val_crops = load_val_crops(val_dir, EVAL_N, device)
    train_loader = build_train_loader(train_dir)
    hires_loader = None
    if HIRES_TAIL_STEPS > 0:
        log(f"  building high-res tail loader ({HIRES_PATCH} crops, batch {HIRES_BATCH}) for the "
            f"{HIRES_TAIL_STEPS}-step DISTS finetune ...")
        hires_loader = build_train_loader(train_dir, patch_override=HIRES_PATCH, batch_override=HIRES_BATCH)
    sh = types.SimpleNamespace(lpips_loss=lpips_loss, dists_loss=dists_loss, msssim_loss=msssim_loss,
                               LP=LP, DS=DS, mse=mse, clip_loss=clip_loss,
                               val_crops=val_crops, train_loader=train_loader,
                               hires_loader=hires_loader, train_dir=train_dir)

    lambdas = FULL_RUN_LAMBDAS if FULL_RUN_LAMBDAS else [LAMBDA_RATE]
    log(f"FULL RUN PLAN: train {len(lambdas)} checkpoint(s) at lambda_rate {lambdas} "
        f"from warm-start {Path(warm).stem}")
    for lam in lambdas:
        run_full_train(lam, repo, sd_dir, vae_dec, warm, assets, device, sh)
    log("=" * 70)
    log(f"  ALL DONE. Deploy .pkl(s) in {OUT_DIR}/AEIC_ME_lovif_ft*.pkl  (one per lambda: "
        + ", ".join(f"ft{int(l)}" for l in lambdas) + ")")
    log("  -> Save Version, make a Kaggle Dataset, attach to stablecodec.ipynb, add each as an")
    log("     AEIC_VARIANTS entry (codec_type 'AEIC-ME'), re-run the knapsack over the richer menu.")
    log("=" * 70)
    return 0


# =============================================================================
#  FULL TRAINING RUN for ONE lambda_rate (called once per FULL_RUN_LAMBDAS entry). Builds a
#  FRESH trainer (warm-started from `warm`) + FRESH optimizer + FRESH GAN discriminator, so the
#  two checkpoints are independent. Resumable via a PER-LAMBDA state file (train_state_ft{lam}.pt)
#  so a single launch trains both and survives the 12h Kaggle cap. The build->train->del->rebuild
#  cycle across lambdas is the same pattern run_sweep already uses successfully.
# =============================================================================
def run_full_train(lam, repo, sd_dir, vae_dec, warm, assets, device, sh):
    import torch
    log("=" * 70)
    log(f"FULL RUN — lambda_rate={lam}  (deploy -> AEIC_ME_lovif_ft{int(lam)}.pkl)")
    mse, clip_loss = sh.mse, sh.clip_loss
    lpips_loss, dists_loss, msssim_loss = sh.lpips_loss, sh.dists_loss, sh.msssim_loss
    LP, DS, val_crops, train_loader = sh.LP, sh.DS, sh.val_crops, sh.train_loader
    hires_loader = sh.hires_loader
    total_steps = MAX_STEPS + HIRES_TAIL_STEPS   # 512 main phase, then 1024 high-res tail

    net, trainable = build_trainer(repo, sd_dir, vae_dec, warm, device)

    # fresh GAN (own discriminator) per generator checkpoint
    gan = None
    if LAMBDA_ADV > 0 and (assets["dinov3_snapshot"] is not None or assets["dinov2"] is not None):
        gan = build_vision_aided_gan(assets, device)
    if gan is None:
        log("  ADVERSARIAL term OFF — full perceptual loss only (rate+L2+MSSSIM+DISTS+LPIPS"
            + ("+CLIP" if clip_loss else "") + ").")
    g_loss = d_loss = None; disc_params = []
    if gan is not None:
        g_loss, d_loss, disc_params = gan

    opt = torch.optim.AdamW(trainable, lr=LR)
    Dopt = torch.optim.AdamW(disc_params, lr=LR_DISC) if disc_params else None

    # ---- resume (survive the 12h Kaggle cap) -- PER-LAMBDA state file ----
    step0 = 0
    state_p = Path(OUT_DIR) / f"train_state_ft{int(lam)}.pt"
    if state_p.exists():
        try:
            st = torch.load(state_p, map_location="cpu")
            _apply_ckpt(net, st["model"], what="resume"); opt.load_state_dict(st["opt"]); step0 = int(st["step"])
            if Dopt is not None and st.get("disc") is not None:
                for p, v in zip(disc_params, st["disc"]): p.data.copy_(v.to(p.device))
                Dopt.load_state_dict(st["Dopt"])
            log(f"  RESUMED from {state_p.name} at step {step0}")
        except Exception as e:
            log(f"  (resume failed, starting fresh: {e})")
    if step0 >= total_steps:
        log(f"  lambda={lam} already complete (step {step0} >= {total_steps}) -- skipping.")
        del net, opt, Dopt; gc.collect(); torch.cuda.empty_cache()
        return

    def lr_at(step):
        if step >= MAX_STEPS: return LR_FLOOR   # high-res tail runs at the LR floor
        c = 0.5 * (1 + math.cos(math.pi * step / MAX_STEPS))
        return LR_FLOOR + (LR - LR_FLOOR) * c

    def adv_w(step):
        return LAMBDA_ADV * min(1.0, step / max(1, GAN_WARMUP)) if gan is not None else 0.0

    def save_all(step, tag=""):
        pkl = Path(OUT_DIR) / f"AEIC_ME_lovif_ft{int(lam)}{tag}.pkl"
        _save_deploy(net, pkl)
        sd = {"step": step, "model": _deploy_sd(net), "opt": opt.state_dict()}
        if Dopt is not None:
            sd["disc"] = [p.detach().cpu() for p in disc_params]; sd["Dopt"] = Dopt.state_dict()
        torch.save(sd, state_p)
        log(f"  saved deploy -> {pkl.name}  | resume -> {state_p.name}  (step {step})")

    @torch.no_grad()
    def evaluate(step):
        if not val_crops: return
        net.eval()
        accs = {k: 0.0 for k in ("bpp", "psnr", "lpips", "dists")}
        for x in val_crops:
            x_hat, RLO, res = net.codec(x, ori_h=PATCH, ori_w=PATCH)
            rec = net.vae.decoder(net.unet(x_hat) + res).clamp(-1, 1)
            x01, r01 = (x * 0.5 + 0.5), (rec * 0.5 + 0.5)
            #  Score the 8-bit PNG the real submission writes, not the float tensor: the
            #  continuous->uint8 round-trip adds quantization noise the live eval otherwise hides.
            if EVAL_PNG_ROUNDTRIP:
                r01 = (r01.clamp(0, 1) * 255.0).round() / 255.0
                rec = r01 * 2 - 1
            accs["bpp"]   += float(RLO.quantized_total_bpp)
            accs["psnr"]  += float(10 * torch.log10(1.0 / mse(r01, x01)))
            accs["lpips"] += float(LP(rec, x).mean())
            accs["dists"] += float(DS(r01.clamp(0, 1), x01.clamp(0, 1)).mean())
        n = len(val_crops); a = {k: v / n for k, v in accs.items()}
        proxy = a["psnr"] + 20 * (1 - a["lpips"]) + 25 * (1 - a["dists"])
        flag = "  <-- near target" if abs(a["bpp"] - TARGET_BPP) < 0.0010 else ""
        log(f"  [eval l{int(lam)} s{step}] bpp={a['bpp']:.5f} (tgt {TARGET_BPP})  PSNR={a['psnr']:.2f}  "
            f"LPIPS={a['lpips']:.4f}  DISTS={a['dists']:.4f}  proxy~{proxy:.3f}{flag}")
        net.codec.train()

    # ---- training loop ----
    tail_note = f" + {HIRES_TAIL_STEPS}@{HIRES_PATCH} tail" if (HIRES_TAIL_STEPS > 0 and hires_loader) else ""
    log(f"STEP 4 — finetune l={lam}: steps {step0}->{total_steps} ({MAX_STEPS}@{PATCH}{tail_note})  "
        f"lr {LR}->{LR_FLOOR}  lambdas(rate={lam} charb={LAMBDA_L2} msssim={LAMBDA_MSSSIM} "
        f"dists={LAMBDA_DISTS} lpips={LAMBDA_LPIPS} clip={LAMBDA_CLIP if clip_loss else 0} "
        f"adv={LAMBDA_ADV if gan else 0})")
    net.codec.train()
    #  Step-driven loop (cycling iterators) so the 512 main phase flows straight into the 1024
    #  high-res tail at step==MAX_STEPS without restarting -- the whole thing shares ONE resumable
    #  step counter, so a crash anywhere (incl. mid-tail) resumes at the right phase.
    lo_iter = _cycle(train_loader)
    hi_iter = _cycle(hires_loader) if (HIRES_TAIL_STEPS > 0 and hires_loader is not None) else None
    step = step0; t_last = time.time()
    if step == 0: evaluate(0)
    while step < total_steps:
        in_tail = step >= MAX_STEPS
        if in_tail and hi_iter is None:        # tail requested but no 1024 loader -> stop cleanly
            break
        if in_tail and step == MAX_STEPS:
            log(f"  >>> HIGH-RES TAIL: {HIRES_TAIL_STEPS} steps @ {HIRES_PATCH} (lr floor {LR_FLOOR}) "
                "-- broadening DISTS/LPIPS texture statistics")
        batch = next(hi_iter) if in_tail else next(lo_iter)
        cur_patch = HIRES_PATCH if in_tail else PATCH
        for g in opt.param_groups: g["lr"] = lr_at(step)
        x = batch.to(device, non_blocking=True).float()

        x_hat, RLO, res = net.codec(x, ori_h=cur_patch, ori_w=cur_patch)
        rec = net.vae.decoder(net.unet(x_hat) + res).clamp(-1, 1)

        l_rate   = RLO.rate_loss
        l_l2     = charbonnier(rec, x)          # Charbonnier anchor (was MSE) -- see charbonnier()
        l_msssim = msssim_loss(rec, x)
        l_dists  = dists_loss(rec, x)
        l_lpips  = lpips_loss(rec, x)
        loss = (lam * l_rate + LAMBDA_L2 * l_l2 + LAMBDA_MSSSIM * l_msssim
                + LAMBDA_DISTS * l_dists + LAMBDA_LPIPS * l_lpips)
        l_clip = None
        if clip_loss is not None:
            l_clip = clip_loss(rec, x); loss = loss + LAMBDA_CLIP * l_clip
        aw = adv_w(step)
        use_gan = gan is not None and aw > 0
        if use_gan:
            # adversarial term is secondary (lambda_adv=0.1) and the only one touching the optional
            # DINOv2 backbone; if its forward throws, self-disable and keep the perceptual loss.
            try:
                loss = loss + aw * g_loss(rec)
            except Exception as e:
                log(f"  !! GAN generator step failed ({type(e).__name__}: {e}) -> "
                    "adversarial term DISABLED, continuing with the perceptual loss")
                gan = None; use_gan = False

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
        opt.step()

        if use_gan:
            try:
                Dopt.zero_grad(set_to_none=True)
                dl = d_loss(rec, x)
                dl.backward()
                torch.nn.utils.clip_grad_norm_(disc_params, GRAD_CLIP)
                Dopt.step()
            except Exception as e:
                log(f"  !! GAN discriminator step failed ({type(e).__name__}: {e}) -> "
                    "adversarial term DISABLED, continuing with the perceptual loss")
                gan = None

        step += 1
        if step % LOG_EVERY == 0:
            dt = (time.time() - t_last) / LOG_EVERY; t_last = time.time()
            extra = f" CLIP={float(l_clip):.4f}" if l_clip is not None else ""
            extra += f" adv_w={aw:.3f}" if gan is not None else ""
            ph = f" [tail@{cur_patch}]" if in_tail else ""
            log(f"  step {step:5d}/{total_steps}{ph}  loss={float(loss):.4f}  rate(bpp)={float(l_rate):.5f}  "
                f"Charb={float(l_l2):.4f}  MSSSIM={1-float(l_msssim):.4f}  DISTS={float(l_dists):.4f}  "
                f"LPIPS={float(l_lpips):.4f}{extra}  lr={lr_at(step):.1e}  {dt:.2f}s/it")
        if step % EVAL_EVERY == 0: evaluate(step)
        if step % CKPT_EVERY == 0:
            save_all(step); gc.collect(); torch.cuda.empty_cache()

    evaluate(step)
    save_all(step, tag="_final")
    log(f"  lambda={lam} DONE -> AEIC_ME_lovif_ft{int(lam)}_final.pkl")
    del net, opt, Dopt; gc.collect(); torch.cuda.empty_cache()


# ---- save/load helpers in the repo's exact save_model() format --------------
def _codec_save_name(k):
    """Rename a live (compressai 1.2.8) codec key BACK to the 1.2.6 EntropyBottleneck naming
    (matrices.{i}->_matrix{i}, etc.) that the public AEIC checkpoints AND stablecodec's
    vendored PracticalEntropyBottleneck use. This makes our deploy .pkl a byte-format DROP-IN
    for stablecodec (loads into its practical codec with zero changes). _apply_ckpt's remap is
    bidirectional, so our own resume reload of these 1.2.6-named keys also works."""
    m = re.match(r"^(.*entropy_bottleneck\.)(matrices|biases|factors)\.(\d+)$", k)
    if m:
        pre, plural, idx = m.groups()
        return pre + {"matrices": "_matrix", "biases": "_bias", "factors": "_factor"}[plural] + idx
    return k

def _deploy_sd(net):
    sd = {}
    sd["state_dict_unet"] = {k: v.detach().cpu() for k, v in net.unet.state_dict().items()
                             if "lora" in k or "conv_in" in k or "conv_out" in k}
    sd["state_dict_codec"] = {_codec_save_name(k): v.detach().cpu()
                              for k, v in net.codec.state_dict().items()}
    return sd

def _save_deploy(net, path):
    import torch
    torch.save(_deploy_sd(net), str(path))

def _eb_remap_candidates(k):
    """Bridge the compressai EntropyBottleneck parameter rename between the version that
    SAVED the public AEIC checkpoints (1.2.6, register_parameter: _matrix{i}/_bias{i}/
    _factor{i}) and the version we RUN (1.2.8, nn.ParameterList: matrices.{i}/biases.{i}/
    factors.{i}). Returns candidate source keys (incl. k itself) to look up in a checkpoint.
    The forward math is byte-identical across these versions, so this is a pure rename."""
    cands = [k]
    m = re.match(r"^(.*entropy_bottleneck\.)(matrices|biases|factors)\.(\d+)$", k)
    if m:
        pre, plural, idx = m.groups()
        cands.append(pre + {"matrices": "_matrix", "biases": "_bias", "factors": "_factor"}[plural] + idx)
    m = re.match(r"^(.*entropy_bottleneck\.)_(matrix|bias|factor)(\d+)$", k)
    if m:
        pre, sing, idx = m.groups()
        cands.append(pre + {"matrix": "matrices", "bias": "biases", "factor": "factors"}[sing] + "." + idx)
    return cands


def _apply_ckpt(net, ckpt, what="checkpoint"):
    """Load {state_dict_codec, state_dict_unet} into the live trainer. Used BOTH for the
    public-checkpoint warm-start (compressai 1.2.6 naming -> remapped) and for resume from
    our own saved state (1.2.8 naming -> remap is a no-op). FAILS LOUD rather than silently
    leaving weights at random init: every non-empty codec tensor and every LoRA/conv_in/
    conv_out unet tensor MUST be matched (with the right shape), else RuntimeError. Empty
    codec buffers (_offset/_quantized_cdf/_cdf_length/scale_table -- regenerated by update(),
    unused in training) are allowed to be absent."""
    ck_codec = ckpt["state_dict_codec"]
    cw = net.codec.state_dict()
    unresolved, shape_bad, remapped = [], [], 0
    for k in cw.keys():
        src = next((c for c in _eb_remap_candidates(k) if c in ck_codec), None)
        if src is None:
            if cw[k].numel() == 0:
                continue                       # update()-regenerated buffer; irrelevant to training
            unresolved.append(k); continue
        if tuple(ck_codec[src].shape) != tuple(cw[k].shape):
            shape_bad.append((k, src, tuple(ck_codec[src].shape), tuple(cw[k].shape))); continue
        cw[k] = ck_codec[src]
        if src != k: remapped += 1
    if shape_bad:
        raise RuntimeError(f"{what}: codec shape mismatch (unexpected compressai/arch diff):\n  "
                           + "\n  ".join(f"{k} <- {s}: ckpt{cs} vs model{ms}" for k, s, cs, ms in shape_bad[:12]))
    if unresolved:
        raise RuntimeError(
            f"{what}: codec keys absent from checkpoint (would stay random-init):\n  "
            + "\n  ".join(unresolved[:20])
            + f"\n  (checkpoint has {len(ck_codec)} codec keys; sample: {list(ck_codec)[:6]})")
    net.codec.load_state_dict(cw)

    ck_unet = ckpt["state_dict_unet"]
    uw = net.unet.state_dict()
    miss_u = []
    for k in uw.keys():
        if "lora" in k or "conv_in" in k or "conv_out" in k:
            if k in ck_unet and tuple(ck_unet[k].shape) == tuple(uw[k].shape):
                uw[k] = ck_unet[k]
            else:
                miss_u.append(k)
    if miss_u:
        raise RuntimeError(f"{what}: unet LoRA/conv_in/conv_out keys missing or shape-mismatched "
                           f"(peft version skew vs the checkpoint?):\n  " + "\n  ".join(miss_u[:20]))
    net.unet.load_state_dict(uw)
    log(f"  {what}: loaded codec ({len(cw)} keys, {remapped} entropy-bottleneck remapped 1.2.6->1.2.8) "
        f"+ unet LoRA/conv_in/conv_out")


if __name__ == "__main__":
    raise SystemExit(main())

# =============================================================================
#  OFFLINE STAGING (RTX PRO 6000, internet OFF) — attach these datasets:
#    1. AEIC staging dataset   (AEIC/ + sd-turbo/ + halfDecoder.ckpt + wheels/)
#                               -> lovif-stage-aeic-assets.py / lovif-stage-wheels-only.py
#    2. torch-hub backbones    (alexnet for LPIPS, vgg16 for DISTS)
#                               -> lovif-stage-metric-backbones.py
#    3. your AEIC checkpoint set (must contain AEIC_ME_ft16.pkl warm-start)
#    4. the LoViF `dataset_train` dataset (14,136 pngs) — already a Kaggle dataset
#    5. (NO-SACRIFICE EXTRAS) the T4-staged train-assets dataset for the GAN+CLIP:
#                               -> lovif-stage-train-assets.py   (DINOv2 + CLIP ViT-B/32)
#       Without #5 the run still does the full perceptual loss; WITH #5 it adds the
#       vision-aided adversarial term (the realism lever) and the CLIP term — i.e.
#       the complete finetune.py recipe, fully offline.
#
#  HITTING 0.008 bpp: watch the [eval] `bpp` line; raise LAMBDA_RATE (28->32->40)
#  if bpp is high, lower if over-smoothed. Final budget is still enforced by the
#  stablecodec knapsack.
#
#  RESUMING ACROSS THE 12h CAP: OUT_DIR/train_state.pt is saved every CKPT_EVERY
#  steps. Save OUT_DIR as a dataset and attach it back next session — the run finds
#  train_state.pt and resumes from the saved step automatically.
# =============================================================================

[     2.8s] STEP 1 — environment setup (offline-first)
[     2.8s]   installed torch_geometric stub finder (compressai's unused point-cloud deps)
[     2.9s]   AEIC source: OK -> /kaggle/input/datasets/ahnaftahmid24/lovif-aeic-offline/results/lovif_aeic_offline/AEIC
[    22.9s]   OFFLINE wheelhouse(s): ['/kaggle/input/datasets/ahnaftahmid24/lovif-aeic-offline/results (3)/wheels']
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
[    31.6s]   TORCH_HOME -> /kaggle/input/datasets/ahnaftahmid24/lovif-aeic-offline/results (2)/torch_hub (staged LPIPS/DISTS backbone weights)


2026-06-30 11:21:39.152078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782818499.300784      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782818499.345918      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782818499.730452      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782818499.730462      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782818499.730464      65 computation_placer.cc:177] computation placer alr

[    45.6s] STEP 2 — models & warm-start checkpoint
[    54.8s]   sd-turbo:    OK /kaggle/input/datasets/ahnaftahmid24/lovif-aeic-offline/results/lovif_aeic_offline/sd-turbo
[    54.8s]   halfDecoder: OK /kaggle/input/datasets/ahnaftahmid24/lovif-aeic-offline/results/lovif_aeic_offline/halfDecoder.ckpt
[    54.8s]   warm-start:  OK /kaggle/input/datasets/tonyironman099/aeic-model-checkpoints/AEIC_ME_ft16.pkl  (candidates: ['AEIC_ME_ft32', 'AEIC_ME_ft16', 'AEIC_ME_ft8', 'AEIC_ME_ft4', 'AEIC_ME_ft2'])
[    56.3s]   loading LPIPS(alex) + DISTS(weights.pt) + MS-SSIM for the training loss ...
[    62.2s]     LPIPS train-blend ON: 0.80*VGG + 0.20*Alex (eval stays pure Alex = board)
[    66.2s]   CLIP loss ON (staged clip-vit-base-patch32)
[    74.3s]   train images: 14136 from /kaggle/input/datasets/tonyironman099/lovif-2026-image-compression/dataset_train (crop=512, batch=2)
[    74.3s]   building high-res tail loader (1024 crops, batch 1) for the 600-step DISTS finetune ...
[    74.4s]   t

SystemExit: 0